In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ot
import imageio
import os

In [ ]:

def make_unbalanced_ot_gif(n_source=50, n_target=50, filename='unbalanced_ot.gif'):
    
    np.random.seed(42)

    # Source: Two clusters (Total mass will be normalized to 1.0)
    mu_s1 = np.array([0, 0])
    mu_s2 = np.array([0, 5])
    cov_s = np.array([[0.5, 0], [0, 0.5]])
    
    xs1 = np.random.multivariate_normal(mu_s1, cov_s, n_source // 2)
    xs2 = np.random.multivariate_normal(mu_s2, cov_s, n_source // 2)
    xs = np.vstack((xs1, xs2))

    # Target: One cluster moved to the right (Total mass will be 0.5 -> UNBALANCED)
    # We force the target to have significantly less mass to see the unbalanced effect
    mu_t = np.array([6, 5])
    cov_t = np.array([[1.0, 0], [0, 2.0]])
    xt = np.random.multivariate_normal(mu_t, cov_t, n_target)

    # --- 2. Define Weights (Unbalanced) ---
    a = np.ones(n_source) / n_source 
    b = np.ones(n_target) / n_target

    print(f"Source Mass: {np.sum(a):.2f}")
    print(f"Target Mass: {np.sum(b):.2f}")

    # --- 3. Compute Unbalanced OT (Sinkhorn) ---
    # Cost matrix (Euclidean distance squared)
    M = ot.dist(xs, xt)
    M /= np.max(M)
    
    # Epsilon (regularization) and Alpha (marginal relaxation)
    epsilon = 0.01
    alpha = 0.1  # Lower alpha = more unbalanced behavior (more mass destruction allowed)

    # Solve using Sinkhorn-Knopp Unbalanced
    # Note: method='sinkhorn' or directly calling sinkhorn_unbalanced
    gamma = ot.sinkhorn_unbalanced(a, b, M, epsilon, alpha)

    # --- 4. Prepare Animation Data ---
    # We treat every entry in gamma as a particle moving from i to j
    # Filter out very small connections for performance
    threshold = 1e-8
    rows, cols = np.where(gamma > threshold)
    weights = gamma[rows, cols]

    print(gamma.shape)
    print(gamma)
    print(weights.shape)
    
    # Scale weights for visualization (marker size)
    # We scale them up so they are visible in the plot
    scale_factor = 10 / np.max(weights)
    sizes = weights * scale_factor

    start_pos = xs[rows]
    end_pos = xt[cols]

    # --- 5. Generate Frames ---
    frames = []
    n_frames = 40
    
    print("Generating frames...")
    
    for t in np.linspace(0, 1, n_frames):
        fig, ax = plt.subplots(figsize=(8, 6))
        
        # Linear interpolation: pos(t) = (1-t)*start + t*end
        current_pos = (1 - t) * start_pos + t * end_pos
        
        # Plot "Ghosts" of source and target for reference
        ax.scatter(xs[:, 0], xs[:, 1], c='blue', alpha=0.1, label='Source')
        ax.scatter(xt[:, 0], xt[:, 1], c='red', alpha=0.1, label='Target')
        
        # Plot the moving mass
        # We use the calculated sizes to show how much mass is moving on this path
        # Color shifts from Blue (start) to Red (end)
        color_val = np.zeros((len(current_pos), 4))
        color_val[:, 0] = t # R
        color_val[:, 2] = 1 - t # B
        color_val[:, 3] = 0.6 # Alpha
        
        ax.scatter(current_pos[:, 0], current_pos[:, 1], s=sizes, c=color_val, edgecolors='none')

        ax.set_title(f"Unbalanced OT Transport (t={t:.2f})")
        ax.set_xlim(-3, 10)
        ax.set_ylim(-3, 8)
        ax.legend(loc='upper right')
        ax.grid(True, linestyle='--', alpha=0.3)

        # Save frame to buffer

        fig.canvas.draw()
        image = np.asarray(fig.canvas.buffer_rgba())        
        image = image[:, :, :3]
        frames.append(image)
        plt.close(fig)

    # --- 6. Save GIF ---
    imageio.mimsave(filename, frames, fps=15)
    print(f"Done! GIF saved to {os.getcwd()}/{filename}")

In [33]:

make_unbalanced_ot_gif()

Source Mass: 1.00
Target Mass: 1.00
(50, 50)
[[4.86126084e-04 7.32384317e-07 4.70929132e-08 ... 1.02585128e-09
  7.74874849e-05 8.85430033e-09]
 [4.96209227e-04 1.15194679e-05 1.45578328e-06 ... 7.56640747e-08
  3.48579682e-04 3.74435405e-07]
 [3.19185175e-04 5.57724234e-07 1.31070236e-08 ... 2.75653466e-10
  2.94633720e-05 8.36988498e-09]
 ...
 [1.23065767e-05 4.32467185e-04 4.34604225e-05 ... 1.97322461e-05
  1.34775768e-04 2.72787759e-04]
 [7.53102361e-07 3.61284861e-04 1.49237887e-04 ... 1.65275238e-04
  5.39843798e-05 5.20317093e-04]
 [1.97261849e-06 4.64456878e-04 1.12799547e-04 ... 9.68687178e-05
  7.76484051e-05 5.46555799e-04]]
(1772,)
Generating frames...
Done! GIF saved to /home/azweig/projects/finfm/unbalanced_ot.gif
